# References Crawling: Statistics and Analysis

This chapter covers the reference-crawling component of Milestone 1: retrieving
reference metadata from Semantic Scholar (S2) for the assigned arXiv
publications. Section 1 defines what is counted, Sections 2–3 explain how the
crawler works and handles errors, and Section 4 reports the statistics.

## 1. Scope and definitions

### 1.1 Unit of analysis

- A **publication** is one base arXiv ID from `papers_index.json`. Its versions
  share one reference snapshot (Section 2.1), so versions do not multiply the
  publication count or the reference count.
- A **reference entry** is one item in a publication's reference list. Entries
  without an arXiv ID or an S2 ID are kept and counted.
- A **publishable reference row** is an entry with a non-null `ref_arxiv_id`;
  only these become rows of the final `references` table. Counts are entries,
  not unique cited papers.


### 1.2 Publication outcomes

| Outcome | Lookup completed | References retrieved | In references-per-paper mean |
| :--- | :---: | :---: | :--- |
| `ok` | Yes | Yes | Positive count |
| `empty_refs` | Yes | Yes | Zero (verified empty list) |
| `not_in_s2` | Yes | No | Excluded |
| Pending (API error, `count_mismatch`, missing/invalid record) | No | No | Excluded |

A verified empty list is a valid result; an API failure is not an empty
bibliography.


### 1.3 Metrics

Let **N** be the number of selected publications, **R** = `ok` + `empty_refs`
(references retrieved) and **C** = R + `not_in_s2` (lookups completed).

| Metric | Calculation |
| :--- | :--- |
| Lookup completion rate | C / N |
| Reference retrieval success rate | R / N |
| Mean references per publication | Σ `n_refs` / R (empty lists count as 0) |
| Mean per-paper S2-ID resolution rate | mean over papers with `n_refs` > 0 of (entries with S2 ID / `n_refs`) |
| Pooled S2-ID resolution rate | Σ entries with S2 ID / Σ `n_refs` |
| arXiv ID coverage | Σ entries with `ref_arxiv_id` / Σ `n_refs` |
| Publishable reference rows | Σ entries with `ref_arxiv_id` |

The S2-ID resolution rate is used as the "success rate for scraping reference
metadata": an entry with an S2 ID was matched to a paper in the S2 graph. A zero
denominator is shown as **N/A**. Performance metrics (time, RAM, disk) are
defined in the Performance chapter.

`ref_submission_date` holds the `publicationDate` returned by S2; it is not an
independently verified arXiv submission date.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from itables import init_notebook_mode, show as _itshow
    init_notebook_mode(all_interactive=False, connected=False)
except ImportError:  # fallback: plain pandas table
    _itshow = None
try:
    from IPython.display import display
except ImportError:
    display = print

# Data exported from the official crawl run (copied into the book repository).
DATA_DIR = Path(os.environ.get("REF_REPORT_DATA", "../data/milestone1/references"))
STATS_CSV = DATA_DIR / "papers_stats.csv"            # from crawl_references.py --summarize
SUMMARY_JSON = DATA_DIR / "summary_references.json"  # from crawl_references.py --summarize
RUN_JSON = DATA_DIR / "run_report.json"              # the selected run_<timestamp>.json
TBD = "[TBD]"


def load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def load_stats(path):
    try:
        return pd.read_csv(path, dtype={"arxiv_id": str})
    except (OSError, ValueError):
        return None


def ratio(num, den):
    return num / den if den else None


def pct(x):
    return "N/A" if x is None or pd.isna(x) else f"{x:.2%}"


def num(x, fmt="{:,.0f}"):
    return "N/A" if x is None or pd.isna(x) else fmt.format(x)


def show_table(df, caption):
    if _itshow is not None:
        _itshow(df, caption=caption, index=False, pageLength=10, scrollX=True, order=[])
    else:
        print(caption)
        display(df)


def missing(what):
    print(f"{TBD} {what} not available yet - will be generated from the official Colab run.")


stats = load_stats(STATS_CSV)
summary = load_json(SUMMARY_JSON)
run = load_json(RUN_JSON)


## 2. Crawling workflow

This section describes how the reference crawler in `src/references/` turns the
list of assigned arXiv IDs into one verified `references.json` file per
publication. Error handling and rate limiting are described in Section 3.

| Module | Responsibility |
| :--- | :--- |
| `crawl_references.py` | CLI entry point: selects jobs, splits batches, verifies and commits records, writes the run report |
| `s2_client.py` | HTTP client for Semantic Scholar (S2): request pacing, retries, shared cooldown after HTTP 429 |
| `records.py` | Validates the input index and API responses, normalizes references, builds and validates records |
| `storage.py` | Atomic JSON writes, record paths, refresh flag |
| `metrics.py` | Per-run counters, RAM sampling, `--summarize` statistics |
| `config.py` | Timing, batch size, retry limits, API key loading |
| `diagnose_429.py` | Stand-alone diagnostic: a few controlled requests without retries to investigate HTTP 429 |
| `clean_references.py` | Deletes generated `references.json` and meta files before a clean re-run |

### 2.1 Input and unit of work

The crawler reads `papers_index.json`, produced by entry discovery, which maps
each base arXiv ID to its list of versions:

```json
{
  "2310.12345": ["v1", "v2"],
  "2310.12346": ["v1"]
}
```

`load_index()` stops with an input error (exit code 2) when the file is missing
or not valid JSON, when the root is not an object, when a key is duplicated,
when a key is not a valid base ID (both `YYMM.NNNNN` and the old
`archive/YYMMNNN` scheme are accepted), or when a version is not of the form
`v<N>`. Duplicate versions are removed and the list is sorted numerically.

**One job = one base ID, not one version.** S2 indexes a *publication*: the
lookup `ARXIV:<base_id>` returns a single reference list and does not
distinguish arXiv versions (the assignment also notes that "version differences
might not be distinguished"). The crawler therefore requests each publication
once and links the resulting snapshot to all of its versions through the field
`applies_to_versions`.


### 2.2 Pipeline overview

```{mermaid}
flowchart TD
    A["papers_index.json"] --> B["Validate index (load_index)"]
    B --> C{"Valid references.json already saved?"}
    C -->|"Yes"| D["Reuse record (resume)"]
    C -->|"No / --force"| E["Split into batches of 200"]
    E --> F["POST /paper/batch"]
    F --> G{"Entry for each paper"}
    G -->|"null"| H["status = not_in_s2"]
    G -->|"count matches and fewer than 500 refs"| I["Build record (count_match)"]
    G -->|"count mismatch or 500+ refs"| J["Paginate /paper/ARXIV:id/references"]
    J --> K{"Paged total = referenceCount?"}
    K -->|"Yes"| L["Build record (paged_count_match)"]
    K -->|"No"| M["Error: paper stays pending"]
    H --> N["Atomic write references.json"]
    I --> N
    L --> N
    N --> O["Append crawl_log.jsonl"]
    M --> O
    O --> P["Final run_<timestamp>.json"]
```

1. **Resume selection.** For each publication, the saved record (if any) is
   checked with `is_valid_record()`. Valid records are reused without an API
   call; if only the version list has changed, `applies_to_versions` is updated
   locally. Everything else goes to the to-do list.
2. **Batching.** The to-do list is split into batches of `BATCH_SIZE = 200`
   papers (the endpoint accepts at most 500).
3. **Batch request.** One `POST` request per batch retrieves the references of
   all papers in it (Section 2.3).
4. **Verification and classification.** Each entry is checked for completeness
   (Section 2.4) and classified as `ok`, `empty_refs`, `not_in_s2`, or left
   pending (outcomes defined in Section 1.2).
5. **Commit.** Each verified record is written immediately and atomically, so
   results already committed survive an interruption later in the batch
   (Section 2.6).
6. **Logging.** One line is appended to `crawl_log.jsonl` after each batch;
   one `run_<timestamp>.json` report is written at the end of the run.


### 2.3 Batch request

```text
POST https://api.semanticscholar.org/graph/v1/paper/batch
     ?fields=title,year,externalIds,referenceCount,
             references.paperId,references.title,references.authors,
             references.year,references.publicationDate,
             references.venue,references.externalIds
body: {"ids": ["ARXIV:2310.12345", "ARXIV:2310.12346", ...]}
```

The batch endpoint was chosen over one `GET` per paper because the crawler
paces requests at one every 10 seconds (Section 3). With *N* publications,
the per-paper approach needs at least *N* requests, while the batch approach
needs ⌈*N*/200⌉ requests in the normal case.

> With N = **[TBD]** assigned publications, this means **[TBD]** batch
> requests instead of **[TBD]** individual requests.

Fields are requested flat (`references.authors`, not `authors.name`), as the
assignment requires. The response is a list aligned with the order of the IDs
sent, with `null` for papers S2 cannot find. Before any entry is used, the
client checks that the list length equals the number of IDs sent and that each
entry has the expected types (`references` is a list, `referenceCount` is a
non-negative integer, each reference field is a string, integer or null).
A response that fails these checks is treated as invalid and retried
(Section 3).


### 2.4 Completeness verification

A reference list embedded in a batch response could be incomplete, and an
incomplete list must not be saved as if it were the full bibliography. The
crawler therefore compares the list with the `referenceCount` returned for the
same paper:

- **Accepted directly** (`verification.method = "count_match"`) when
  `referenceCount == len(references)` *and* the list has fewer than
  `TRUNCATION_CHECK_MIN_REFS = 500` entries.
- **Re-fetched by pages** otherwise: the crawler calls
  `GET /paper/ARXIV:{id}/references?limit=1000&offset=...` and follows the
  `next` offset until it is absent. If `referenceCount` was missing, it is
  looked up with `GET /paper/ARXIV:{id}?fields=paperId,referenceCount`. The
  record is accepted (`"paged_count_match"`) only if the total number of paged
  references equals `referenceCount`.
- **Left pending** if the counts still differ (`count_mismatch`). Nothing is
  written for that paper, and the next run will try it again.

The threshold of 500 is a conservative choice, not a documented S2 limit: large
lists are always re-checked by pages even when the counts agree. Each page is
also validated: `data` must be a list, `next` must increase and must not repeat
an earlier offset, and an empty page that still has `next` is rejected. These
checks prevent an infinite pagination loop.

A paper whose batch entry is `null` is recorded as `not_in_s2` with
`verification.method = "not_found"`. This is a completed lookup, not an error.


### 2.5 Reference normalization and record format

Every reference entry is normalized by `normalize_ref()`:

| S2 field | Record field | Note |
| :--- | :--- | :--- |
| `externalIds.ArXiv` | `ref_arxiv_id` | Version suffix removed; if absent, extracted from a DOI of the form `10.48550/arXiv.<id>`; origin stored in `arxiv_id_source` |
| `paperId` | `ref_s2_id` | Semantic Scholar ID |
| `title` | `ref_title` | |
| `authors[].name` | `ref_authors` | List of strings; null names dropped |
| `publicationDate` | `ref_submission_date` | Publication date from S2, not a verified arXiv submission date |
| `year`, `venue` | `ref_year`, `ref_venue` | Extra metadata |
| `externalIds` | `ref_external_ids` | Full dictionary kept (DOI, CorpusId, ...) |

All entries are kept, including those without an arXiv ID or without an S2 ID,
as the assignment requires. Old-scheme identifiers such as `hep-th/9901001` are
kept unchanged.

Structure of one record (illustrative values):

```json
{
  "arxiv_id": "2310.12345",
  "s2_paper_id": "…",
  "s2_reference_count": 42,
  "status": "ok",
  "applies_to_versions": ["2310.12345v1", "2310.12345v2"],
  "n_refs": 42,
  "n_refs_with_arxiv": 30,
  "n_refs_without_s2_id": 1,
  "references": [
    {
      "ref_arxiv_id": "1706.03762",
      "arxiv_id_source": "externalIds",
      "ref_title": "Attention Is All You Need",
      "ref_authors": ["Ashish Vaswani", "..."],
      "ref_submission_date": "2017-06-12",
      "ref_year": 2017,
      "ref_venue": "Neural Information Processing Systems",
      "ref_s2_id": "…",
      "ref_external_ids": {"ArXiv": "1706.03762", "DOI": "…"}
    }
  ],
  "fetched_at": "2026-11-08T12:00:00+00:00",
  "verification": {"complete": true, "method": "count_match"}
}
```

For the final `references` table, each entry with a non-null `ref_arxiv_id`
becomes one row keyed by the citing `arxiv_id`; entries without an arXiv ID stay
in the local corpus only.


### 2.6 Storage, resume and refresh

**Layout.** Records follow the suggested one-folder-per-publication convention:
the folder name replaces `.` with `-` and `/` with `_`.

```text
data/output/papers/
├── 2310-12345/references.json
├── hep-th_9901001/references.json
└── ...
data/output/papers_references_meta/
├── crawl_log.jsonl          # one line per batch
├── run_<timestamp>.json     # one report per run (counts, timing, RAM, disk)
├── summary_references.json  # from --summarize
└── papers_stats.csv         # from --summarize
```

**Atomic write.** A record is first written to a temporary file in the same
folder, flushed and `fsync`-ed, then moved into place with `os.replace()`.
On Windows, temporary lock errors (WinError 5/32/33) are retried up to six
times. A crash or `Ctrl+C` therefore never leaves a half-written
`references.json`.

**Resume.** Re-running the same command continues where the last run stopped.
`is_valid_record()` reuses a saved record only if its ID matches, its status is
terminal, its counters agree with the stored list, its verification is complete
and it is not flagged `refresh_pending`. Corrupted or incomplete files are
crawled again.

**Forced refresh.** With `--force`, every existing record in a batch is first
flagged `refresh_pending: true` *before* the HTTP request. A successful refresh
replaces the record; if the refresh fails, the old snapshot stays on disk but is
no longer counted as completed.

**Offline summary.** `--summarize` recomputes the statistics from saved records
only, without any API call, and writes `summary_references.json` and
`papers_stats.csv`.

### 2.7 Running the crawler

```bash
python src/references/crawl_references.py data/input/papers_index.json
python src/references/crawl_references.py data/input/papers_index.json --summarize
```

Options: `--limit N` (first N publications), `--batch-size`, `--cycle`,
`--wait-429`, `--force`, `--out`, `--meta-dir`. Exit codes: `0` all selected
papers completed, `1` some papers pending or run stopped by a fatal error,
`2` input error, `130` interrupted by the user.


## 3. Error handling and rate limiting

### 3.1 Request pacing

- **Sequential requests.** The client uses one `requests.Session` and sends one
  request at a time; there is no concurrency.
- **Fixed cycle.** A request may start only `CYCLE_SECONDS = 10` seconds after
  the *start* of the previous one, so a slow response does not add a further
  10-second delay on top.
- **Margin over the documented limits.** S2 allows 1 request/second with an API
  key and about 100 requests per 5 minutes (≈ 0.33 request/second) for the
  shared unauthenticated pool. One request every 10 seconds (0.1 request/second)
  is 10 times below the keyed limit and about 3 times below the unauthenticated
  one.
- **Identification.** The API key is read from `.env` (`S2_API_KEY`) and sent in
  the `x-api-key` header; it is never written to logs or reports. Requests carry
  a descriptive `User-Agent`.


### 3.2 Error classification

Every response goes through `S2Client._request()`, which decides whether to
accept, retry, or stop:

| Situation | Handling | Retry limit (per request) | When retries run out |
| :--- | :--- | :---: | :--- |
| HTTP 200, valid JSON and schema | Accept | – | – |
| HTTP 200, invalid JSON or schema | Back off 4 s → 8 s → 16 s, resend | 3 | Papers of the batch stay pending (`invalid_response`) |
| HTTP 429 Too Many Requests | Shared cooldown of max(60 s, `Retry-After`), resend the same request | 8 | Papers stay pending (`retryable`) |
| HTTP 503 | Shared cooldown of max(backoff, `Retry-After`) | 3 | Papers stay pending |
| Other HTTP 5xx | Back off 4 s → 8 s → 16 s | 3 | Papers stay pending |
| Network error / timeout (120 s) | Back off 4 s → 8 s → 16 s | 3 | Papers stay pending |
| HTTP 400 "response too large" | Split the batch in half recursively; a single paper is fetched by pages | – | – |
| HTTP 401 / 403 | Stop the run (`FatalError`): API key or permission problem | 0 | Exit code 1 |
| Other HTTP 4xx | Stop the run: the request itself is wrong, retrying cannot help | 0 | Exit code 1 |
| `referenceCount` ≠ references received | Not committed (Section 3.5) | – | Paper stays pending (`count_mismatch`) |

The rules follow one idea: **retry what is temporary, stop on what is
permanent, and never save an unverified result.**

- 429 and other errors have separate retry counters, so a burst of 429 does not
  use up the budget for network errors and vice versa.
- A retry resends the *same* request, so a batch is not lost after a transient
  failure.
- When retries run out, all papers of that batch are recorded in the run
  report's `errors` list with an error code, and the crawler moves on to the
  next batch. Pending papers are picked up automatically by the next run
  (resume, Section 2.6).
- Only configuration problems (401/403, malformed requests) stop the whole run,
  because every following request would fail the same way. Records already
  committed are kept.


### 3.3 Shared cooldown after HTTP 429

After a 429 response the client sets a **global** cooldown deadline
(`_cooldown_until`) instead of only delaying the failed request:

1. The wait is `max(WAIT_429_SECONDS = 60, Retry-After)`. `Retry-After` is read
   either as seconds or as an HTTP date; an invalid value is ignored.
2. No request of any kind (batch, page, metadata lookup) is sent before the
   deadline. The 10-second pacing still applies after the cooldown.
3. The cooldown is also set by the last 429 that exhausts the retry budget, so
   the next batch does not hit the server immediately.

In the worst case one request waits 8 × 60 s = 8 minutes in cooldown before it
is given up. All waiting time is recorded separately in the run report
(`sleep_429_sec`, `wait_sec`, `backoff_sec`), which makes its effect on the
running time measurable (Section 3.6).


### 3.4 Interruptions and storage failures

| Event | Behaviour | Exit code |
| :--- | :--- | :---: |
| `Ctrl+C` | Stop; committed records kept (atomic writes); run report still written with status `interrupted` | 130 |
| Cannot write a record (`OSError`) | Stop; earlier records kept | 1 |
| Cannot write `crawl_log.jsonl` or the run report | Warning only; crawling continues | – |
| Unexpected exception | Logged with traceback; committed records kept | 1 |

### 3.5 Reference count mismatch

For some papers S2 returns a `referenceCount` that differs from the number of
references it actually delivers, on both the batch and the paged endpoint. In
the trial runs most differences were a single entry. The crawler cannot tell
which number is correct, so it does not commit these papers; they remain pending
with code `count_mismatch`.

Re-running retries these papers, but if the inconsistency lies in S2's own
data it may persist. They are therefore reported as pending rather than counted
as successful.

> Number of `count_mismatch` papers in the official run: **[TBD]**.


### 3.6 The persistent HTTP 429 problem

**Observation.** Even with 10-second pacing, a 60-second cooldown and an API key,
the crawler still received HTTP 429 often: **[TBD]** of **[TBD]** requests in
the official run (table below).

**Why our request rate is not the likely cause.** The client sends at most one
request every 10 seconds, which is 10 times below the documented keyed limit,
and it never sends requests in parallel. A client-side burst cannot occur
because the pacer and the shared cooldown apply to every request.

**Suspected cause: server-side throttling.** S2 documents that shared capacity
is throttled during heavy use. 429 responses that appear regardless of our own
rate point to throttling or load shedding on the server side, possibly stronger
for large batch requests (200 papers with full reference lists in one
response). This is a hypothesis we cannot verify from the client side.

**Diagnosis tool.** `diagnose_429.py` sends a few controlled requests *without
retries* and prints the status code, response time, size, body excerpt and any
`Retry-After` / `X-RateLimit-*` headers:

1. `GET` one paper → 2. `POST` batches of 10, 50, 200 papers → 3. `GET` one paper again.

| Result | Interpretation |
| :--- | :--- |
| Step 1 already returns 429 | The limit concerns the key/account or the shared pool, not the batch size |
| Step 1 = 200 but large batches return 429 | Large batches are throttled; use a smaller `--batch-size` |
| Step 1 returns 403 | The API key is rejected |

> Diagnostic results: **[TBD — insert the output of `diagnose_429.py`]**.

**What we can and cannot control.**

- *Can:* keep pacing, cool down after every 429, retry with a limit, keep all
  committed work, resume pending papers in a later run, and reduce
  `--batch-size` or increase `--cycle` if the diagnosis points to them.
- *Cannot:* the server's throttling policy and its current load. Since 429
  persisted at a rate already 10 times below the limit, slowing down further is
  unlikely to remove it and would mainly make the run longer.

The 429 problem therefore mainly affects **running time**, not data
correctness: a throttled request is resent, and no unverified result is saved.
The table and chart below show its cost in the selected run.


In [ ]:
if run is None or "requests" not in run:
    missing("Request and wait statistics (run_report.json)")
else:
    rq, wall = run["requests"], run["time"]["total_wall_sec"]
    show_table(pd.DataFrame([
        ("HTTP requests sent", num(rq["requests"])),
        ("HTTP 429 responses retried", num(rq["retries_429"])),
        ("429 retries per request", pct(ratio(rq["retries_429"], rq["requests"]))),
        ("Other retries (5xx, network, schema)", num(rq["retries_other"])),
        ("Total wall time (s)", num(wall, "{:,.1f}")),
        ("Time in 429 cooldown (s)", num(rq["sleep_429_sec"], "{:,.1f}")),
        ("429 cooldown share of wall time", pct(ratio(rq["sleep_429_sec"], wall))),
    ], columns=["Measure", "Value"]), caption="HTTP requests and HTTP 429 in the selected run")

    parts = {
        "Waiting for responses": rq["request_sec"],
        "Pacing (10 s cycle)": rq["wait_sec"],
        "429 cooldown": rq["sleep_429_sec"],
        "Backoff (other errors)": rq["backoff_sec"],
    }
    parts["Local processing / I/O"] = max(0.0, wall - sum(parts.values()))
    fig, ax = plt.subplots(figsize=(7, 2.8))
    left = 0.0
    for label, sec in parts.items():
        ax.barh([0], [sec], left=left, label=f"{label}: {sec / 60:,.1f} min")
        left += sec
    ax.set_yticks([])
    ax.set_xlabel("seconds")
    ax.set_title("Where the wall time of the selected run was spent")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=2, fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()


## 4. Results

### 4.1 Data used

All figures and tables below are computed during book compilation from files
exported after the official run; compiling the book does not call the API.

| File | Produced by | Used for |
| :--- | :--- | :--- |
| `papers_stats.csv` | `crawl_references.py --summarize` | One row per selected publication: status, `n_refs`, `n_refs_with_arxiv`, `n_refs_without_s2_id` |
| `summary_references.json` | `crawl_references.py --summarize` | Cross-check of totals; IDs of pending papers |
| `run_report.json` | The selected `run_<timestamp>.json` | Error codes of pending papers, request and timing statistics |

`--summarize` re-reads every `references.json` and validates it with the same
check used for resume, so only verified records are counted.


In [ ]:
if stats is None:
    missing("papers_stats.csv")
else:
    checks = [("Rows in papers_stats.csv", f"{len(stats):,}"),
              ("Duplicate arxiv_id", f"{stats['arxiv_id'].duplicated().sum():,}")]
    if summary is not None:
        p, r = summary["papers"], summary["references"]
        done = stats[stats["complete"]]
        checks += [("Selected publications in summary", f"{p['selected']:,}"),
                   ("Rows match selected publications", str(len(stats) == p["selected"])),
                   ("Reference totals match summary", str(int(done["n_refs"].sum()) == r["total"]))]
    show_table(pd.DataFrame(checks, columns=["Check", "Result"]), caption="Consistency checks of the exported data")


### 4.2 Publication outcomes


In [ ]:
if stats is None:
    missing("Publication outcomes")
else:
    terminal = ["ok", "empty_refs", "not_in_s2"]
    status = stats["status"].where(stats["status"].isin(terminal), "pending")
    counts = status.value_counts().reindex(terminal + ["pending"], fill_value=0)
    n = len(stats)
    show_table(pd.DataFrame({"Outcome": counts.index, "Publications": counts.values,
                             "Share": [pct(ratio(v, n)) for v in counts.values]}),
               caption=f"Outcome of the {n:,} selected publications")
    fig, ax = plt.subplots(figsize=(6, 3))
    bars = ax.bar(counts.index, counts.values, color=["#4c72b0", "#8fa8d0", "#dd8452", "#c44e52"])
    ax.bar_label(bars, labels=[f"{v:,}" for v in counts.values], fontsize=8)
    ax.set_ylabel("publications")
    ax.set_title("Publication outcomes")
    plt.tight_layout()
    plt.show()


### 4.3 Key metrics

Definitions are given in Section 1.3.


In [ ]:
if stats is None:
    missing("Key metrics")
else:
    n = len(stats)
    ret = stats[stats["status"].isin(["ok", "empty_refs"])]
    R, C = len(ret), len(ret) + int((stats["status"] == "not_in_s2").sum())
    total = ret["n_refs"].sum()
    resolved = total - ret["n_refs_without_s2_id"].sum()
    nonempty = ret[ret["n_refs"] > 0]
    per_paper = ((nonempty["n_refs"] - nonempty["n_refs_without_s2_id"]) / nonempty["n_refs"]).mean()
    metrics = pd.DataFrame([
        ("Selected publications (N)", num(n)),
        ("Lookups completed (C)", num(C)),
        ("References retrieved (R)", num(R)),
        ("Lookup completion rate (C / N)", pct(ratio(C, n))),
        ("Reference retrieval success rate (R / N)", pct(ratio(R, n))),
        ("Reference entries", num(total)),
        ("Mean references per publication", num(ratio(total, R), "{:,.2f}")),
        ("Median references per publication", num(ret["n_refs"].median() if R else None, "{:,.1f}")),
        ("Mean per-paper S2-ID resolution rate", pct(per_paper if len(nonempty) else None)),
        ("Pooled S2-ID resolution rate", pct(ratio(resolved, total))),
        ("arXiv ID coverage", pct(ratio(ret["n_refs_with_arxiv"].sum(), total))),
        ("Publishable reference rows", num(ret["n_refs_with_arxiv"].sum())),
    ], columns=["Metric", "Value"])
    show_table(metrics, caption="Reference crawling metrics")


### 4.4 Distribution of references per publication

The histograms use publications with retrieved references (`ok` and
`empty_refs`). The x-axis of the first chart is cut at the 99th percentile so
that a few very long bibliographies do not hide the main shape; the cut is
stated in the title.


In [ ]:
if stats is None:
    missing("Reference distributions")
else:
    ret = stats[stats["status"].isin(["ok", "empty_refs"])]
    nonempty = ret[ret["n_refs"] > 0]
    if ret.empty:
        missing("Publications with retrieved references")
    else:
        cap = ret["n_refs"].quantile(0.99)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
        axes[0].hist(ret["n_refs"].clip(upper=cap), bins=50, color="#4c72b0")
        axes[0].axvline(ret["n_refs"].mean(), color="black", ls="--", label=f"mean = {ret['n_refs'].mean():.1f}")
        axes[0].axvline(ret["n_refs"].median(), color="grey", ls=":", label=f"median = {ret['n_refs'].median():.0f}")
        axes[0].set(title=f"References per publication (clipped at p99 = {cap:.0f})",
                    xlabel="reference entries", ylabel="publications")
        axes[0].legend(fontsize=8)
        cov = nonempty["n_refs_with_arxiv"] / nonempty["n_refs"]
        axes[1].hist(cov, bins=20, range=(0, 1), color="#dd8452")
        axes[1].set(title="Share of references with an arXiv ID (per publication)",
                    xlabel="arXiv ID coverage", ylabel="publications")
        plt.tight_layout()
        plt.show()


### 4.5 Pending publications

Pending publications have no verified record. The table groups them by error
code from the run report; publications without a code in the report are
listed as `no record`.


In [ ]:
if stats is None:
    missing("Pending publications")
else:
    pending = stats.loc[~stats["status"].isin(["ok", "empty_refs", "not_in_s2"]), "arxiv_id"]
    if pending.empty:
        print("No pending publications.")
    else:
        codes = {e["arxiv_id"]: e["code"] for e in (run or {}).get("errors", [])}
        reasons = pending.map(lambda a: codes.get(a, "no record")).value_counts()
        show_table(pd.DataFrame({"Reason": reasons.index, "Publications": reasons.values}),
                   caption=f"{len(pending):,} pending publications by reason")


### 4.6 Interpretation

- **Coverage of S2.** **[TBD]** of the selected publications were found in S2;
  `not_in_s2` papers are most likely very recent uploads not yet indexed (S2
  indexing can lag by days).
- **Empty reference lists.** **[TBD]** publications have a verified empty list.
  S2 found the paper but did not extract its bibliography, so these are counted
  as successful retrievals with zero references.
- **arXiv coverage.** **[TBD]** of the reference entries carry an arXiv ID and
  can become rows of the final `references` table; the rest cite journals,
  books or other sources and stay in the local corpus.
- **Pending.** The remaining **[TBD]** publications are pending, mainly because
  of `count_mismatch` (Section 3.5).
